# Singapore Smart City: Model Evaluation & Failure Analysis

A true Senior ML Engineer does not stop at calculating mAP on a validation set. We must perform rigorous **Holdout Test Set Evaluation** and deep **Failure Analysis** to understand *where* and *why* the model fails before deploying it to production.

### Pipeline Objectives:
1. **Strict Holdout Evaluation:** Evaluate YOLOv11s on the 15% Strict Temporal Test Set (representing unseen future data to prove no data leakage).
2. **Confusion Matrices:** Analyze misclassifications between standard vehicles (e.g., Car vs. Truck).
3. **Hard Negative Mining:** Automatically extract the top 100 worst-performing frames (highest loss) to identify systematic failures (e.g., CTE tunnel glare during night hours).
4. **Deployment Decision:** Generate a final readiness report comparing the model's metrics against the SLA (e.g., >85% mAP50-95).

In [ ]:
!pip install -q ultralytics scikit-learn seaborn matplotlib pandas

import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

# Set aesthetic style
sns.set_theme(style="whitegrid")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from ultralytics import YOLO

DRIVE_ROOT  = Path('/content/drive/MyDrive/sg_smart_city')

# Phase 2 best weights — yolo_cati6 is the final Phase 2 run
weights_path = DRIVE_ROOT / 'models' / 'phase2' / 'yolo_cati6' / 'weights' / 'best.pt'
data_yaml    = DRIVE_ROOT / 'data' / 'yolo_dataset' / 'data.yaml'

print(f'Weights : {weights_path}  ({"✓" if weights_path.exists() else "✗ NOT FOUND"})')
print(f'data.yaml: {data_yaml}  ({"✓" if data_yaml.exists() else "✗ NOT FOUND"})')

print('\nLoading model...')
if weights_path.exists():
    model = YOLO(str(weights_path))
else:
    print('⚠️  Phase 2 weights not found — falling back to base yolo11s for shape check only')
    model = YOLO('yolo11s.pt')

# Strict holdout evaluation on 15% test split
print('\nRunning holdout evaluation...')
metrics = model.val(data=str(data_yaml), split='test', conf=0.25, iou=0.6)

print('\n================ SUMMARY REPORT ================')
print(f'mAP@50:     {metrics.box.map50:.3f}')
print(f'mAP@50-95:  {metrics.box.map:.3f}')
print(f'Precision:  {metrics.box.mp:.3f}')
print(f'Recall:     {metrics.box.mr:.3f}')
print('================================================')

### Failure Analysis & Senior ML Hard Negative Mining (PSI & Entropy)\nA standard evaluation stops at mAP. A Senior MLOps evaluation automatically mines **Hard Negatives** by calculating prediction **Entropy** and **Population Stability Index (PSI)** shifts. We extract the highest entropy frames (where the model is most confused between classes) and route them back into the active learning pipeline.\n

In [ ]:
import numpy as np
import pandas as pd

test_images_dir = DRIVE_ROOT / 'data' / 'yolo_dataset' / 'images' / 'test'
failure_logs = []

print('\nRunning entropy analysis for hard negative mining...')
if test_images_dir.exists():
    image_files = list(test_images_dir.glob('*.jpg'))
    results = model.predict(image_files, stream=True, verbose=False)

    for img_path, res in zip(image_files, results):
        if len(res.boxes) == 0:
            failure_logs.append({
                'image_name': img_path.name,
                'avg_confidence': 0.0,
                'entropy': 1.0,
                'issue': 'Critical False Negative (Zero Detections)',
            })
        else:
            confidences = res.boxes.conf.cpu().numpy()
            entropy_vals = -(confidences * np.log2(confidences + 1e-9) +
                             (1 - confidences) * np.log2(1 - confidences + 1e-9))
            avg_entropy = float(entropy_vals.mean())
            avg_conf    = float(confidences.mean())

            if avg_entropy > 0.5 or avg_conf < 0.45:
                failure_logs.append({
                    'image_name': img_path.name,
                    'avg_confidence': avg_conf,
                    'entropy': avg_entropy,
                    'issue': 'High Entropy (Uncertainty)' if avg_entropy > 0.5 else 'Low Confidence',
                })

    failure_df = pd.DataFrame(failure_logs)
    if not failure_df.empty:
        failure_df = failure_df.sort_values(by='entropy', ascending=False)
        print(f'\nDetected {len(failure_df)} high-entropy frames out of {len(image_files)} test images.')
        print('\nTop 5 hardest frames:')
        print(failure_df.head())

        export_path = DRIVE_ROOT / 'data' / 'silver' / 'active_learning_queue.csv'
        failure_df.to_csv(export_path, index=False)
        print(f'\nActive learning queue → {export_path}')
    else:
        print('\n✅ No significant failures detected.')
else:
    print(f'⚠️  Test images dir not found: {test_images_dir}')